<a href="https://colab.research.google.com/github/noa-bedoya/boltz-colab/blob/main/BOLTZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boltz

In [ ]:
!rm -rf ~/.boltz/mols.tar ~/.boltz/mols

In [ ]:
!rm -rf ~/.boltz/

In [ ]:
#@markdown ### Configuración de los archivos para ejecutar Boltz
#@markdown Asegurarse de que todo coincide con lo deseado:
nombre_del_trabajo = "prot_xxxx.yaml" #@param {type:"string"}

!boltz predict {nombre_del_trabajo} --use_msa_server --out_dir resultados_boltz


In [ ]:
#@title Visualizar Estructura 3D de Boltz {run: "auto"}
# Primero instalamos la librería gráfica (en silencio)
!pip install -q py3Dmol

import py3Dmol
import glob

#@markdown ### Configuración de los archivos
#@markdown Asegurarse de que los nombres coinciden con tu comando anterior:
directorio_resultados = "./resultados/boltz_results_prot_xxxx/predictions" #@param {type:"string"}
nombre_del_trabajo = "prot_xxxx" #@param {type:"string"}

#@markdown ### Opciones de visualización
color = "pLDDT (Confianza)" #@param ["pLDDT (Confianza)", "rainbow", "chain"]
mostrar_cadenas_laterales = False #@param {type:"boolean"}

def visualizar_boltz():
    # Boltz suele guardar el archivo final en formato .cif, buscamos en la carpeta
    ruta_busqueda = f"{directorio_resultados}/{nombre_del_trabajo}/*_model_*.cif"
    archivos = glob.glob(ruta_busqueda)

    # Si no encuentra un .cif, busca un .pdb por si acaso
    if not archivos:
        archivos = glob.glob(f"{directorio_resultados}/{nombre_del_trabajo}/*.pdb") + glob.glob(f"{directorio_resultados}/{nombre_del_trabajo}/*.cif")

    if not archivos:
        print(f"❌ No se encontró ningún archivo de estructura en: {directorio_resultados}/{nombre_del_trabajo}/")
        print("Asegúrate de que la predicción terminó correctamente.")
        return

    archivo_final = archivos[0]
    print(f"✅ Mostrando estructura: {archivo_final}")

    # Inicializar el visor 3D
    view = py3Dmol.view(width=800, height=500)
    formato = 'cif' if archivo_final.endswith('.cif') else 'pdb'

    with open(archivo_final, 'r') as f:
        view.addModel(f.read(), formato)

    # Configurar el esquema de colores
    if color == "pLDDT (Confianza)":
        # Usa la escala clásica (rojo=malo, azul=excelente) basada en el B-factor
        view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':50,'max':90}}})
    elif color == "rainbow":
        view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
        view.setStyle({'cartoon': {'colorscheme':'chain'}})

    # Dibujar cadenas laterales (los "palitos" de los aminoácidos)
    if mostrar_cadenas_laterales:
        atomos_principales = ['C','O','N']
        view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':atomos_principales,'invert':True}]},
                      {'stick':{'colorscheme':"WhiteCarbon",'radius':0.2}})

    view.zoomTo()
    view.show()

visualizar_boltz()

In [ ]:
#@title Plots de Boltz (PAE,PDE) {run: "auto"}

import numpy as np
import matplotlib.pyplot as plt

#@markdown ### Configuración de los archivos
#@markdown Asegurarse de que los nombres coinciden con tu comando anterior:
directorio_resultados = "./resultados/boltz_results_prot_xxxx/predictions" #@param {type:"string"}
nombre_del_trabajo = "prot_xxxx" #@param {type:"string"}


# Carga tus datos reales
pae_data = np.load(f"{directorio_resultados}/{nombre_del_trabajo}/pae_{nombre_del_trabajo}_model_0.npz")
pae_matrix = pae_data[pae_data.files[0]]

pde_data = np.load(f"{directorio_resultados}/{nombre_del_trabajo}/pde_{nombre_del_trabajo}_model_0.npz")
pde_matrix = pde_data[pde_data.files[0]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de PAE
im1 = ax1.imshow(pae_matrix, cmap='Greens_r', vmin=0, vmax=30) # Verde oscuro = menor error
ax1.set_title('PAE (Predicted Aligned Error)')
ax1.set_xlabel('Residuo')
ax1.set_ylabel('Residuo')
fig.colorbar(im1, ax=ax1, label='Error (Å)')

# Gráfico de PDE
im2 = ax2.imshow(pde_matrix, cmap='magma') # Magma es excelente para ver focos de error de difusión
ax2.set_title('PDE (Predicted Diffusion Error)')
ax2.set_xlabel('Residuo')
ax2.set_ylabel('Residuo')
fig.colorbar(im2, ax=ax2, label='Error de Difusión')

plt.tight_layout()
plt.show()

In [ ]:
import os
from google.colab import files

#@markdown ### Configuración de descargas
#@markdown Pega aquí la ruta exacta den la carpeta:
ruta_carpeta_exacta = "/content/resultados/boltz_results_prot_xxxx" #@param {type:"string"}
nombre_del_zip_final = "mis_resultados_xxxx.zip" #@param {type:"string"}

# Comprobamos primero si la carpeta realmente existe para evitar errores
if os.path.exists(ruta_carpeta_exacta):
    print("¡Carpeta encontrada! Comprimiendo...")
    # Comprimimos la carpeta
    !zip -r {nombre_del_zip_final} {ruta_carpeta_exacta}

    print("Descargando...")
    # Descargamos el archivo
    files.download(nombre_del_zip_final)
else:
    print(f"❌ ERROR: No se encuentra la carpeta en la ruta: {ruta_carpeta_exacta}")
    print("Por favor, asegúrate de haber copiado bien la ruta desde el menú izquierdo.")